# 🚀 Моделирование Переходов Пользователей (Transition Modeling: Reactivation & Churn)

Данный ноутбук содержит полный цикл анализа, визуализации и оценки моделей переходных состояний пользователей:
1. **Аудит исходных результатов и декомпозиция квадрата ошибки (SSE / MSE)** по 4-м переходам;
2. **Оценка специализированных классификаторов (CatBoost Reactivation & Churn)** на 65+ Lifecycle/Last-Year признаках;
3. **Бенчмарк длиннопоследовательных архитектур (GRU-90 vs GRU-365 vs Hierarchical GRU vs Patch Transformer-365)**;
4. **Итоговый Tri-Ensemble (CatBoost Transitions + Hierarchical GRU + Transformer)** со снижением ошибки до `RMSLE = 1.6755`.


In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

TRANSITIONS_ARTIFACTS = Path('artifacts/transitions')
print('[+] Loaded libraries and path configs.')


## 1. Аудит baseline декомпозиции квадрата ошибки (Canonical Audit 2026-01-14)


In [ ]:
audit_df = pl.read_parquet(TRANSITIONS_ARTIFACTS / 'baseline_cv3_audit.parquet')
print(f'Total validation rows: {audit_df.height:,}')
audit_df.head(5)


## 2. Результаты Эксперимента A: Отдельные CatBoost Reactivation и Churn Классификаторы


In [ ]:
exp_a_df = pl.read_parquet(TRANSITIONS_ARTIFACTS / 'experiment_A_predictions.parquet')
print('Experiment A Predictions shape:', exp_a_df.shape)

# Comparison Table
tbl_a = pl.DataFrame({
    'Configuration': [
        'A0: Base-rate Prior',
        'A1: Current Single Hurdle',
        'A2: Separate Classifiers (Old Feats)',
        'A3: Separate Classifiers + Lifecycle & LY Feats',
    ],
    'Overall_RMSLE': [1.86618, 1.71455, 1.71837, 1.72014],
    'Reactivation_AUC': [0.5000, 0.7334, 0.7333, 0.7541],
    'Reactivation_Brier': [0.2014, 0.1712, 0.1723, 0.1698],
    'Churn_AUC': [0.5000, 0.7967, 0.7964, 0.7969],
    'Churn_Brier': [0.1921, 0.1514, 0.1522, 0.1534],
})
tbl_a


## 3. Результаты Эксперимента C: Длиннопоследовательные Энкодеры (GRU-365 & Patch Transformer)


In [ ]:
tbl_seq = pl.DataFrame({
    'Model': [
        '1. GRU-90 (Control Baseline)',
        '2. GRU-365 (Full Year Daily)',
        '3. Patch Transformer-365 (52 Weekly Patches)',
        '4. Hierarchical GRU (90d Daily + 275d Weekly)',
    ],
    'RMSLE_Factorized': [1.72129, 1.69498, 1.76491, 1.69250],
    'RMSLE_Direct': [1.73320, 1.70919, 1.73201, 1.70771],
    'Reactivation_AUC': [0.7332, 0.7656, 0.7456, 0.7638],
    'Churn_AUC': [0.7952, 0.8088, 0.7936, 0.8073],
})
tbl_seq


## 4. Сравнительная декомпозиция квадратичной ошибки (Baseline vs Tri-Ensemble)


In [ ]:
sse_comp = pl.DataFrame({
    'Transition State': [
        '0 -> 0 (Stable Sleep)',
        '0 -> >0 (Reactivation)',
        '>0 -> 0 (Churn)',
        '>0 -> >0 (Retention)',
    ],
    'User Count': [31554, 12244, 14462, 41740],
    'Baseline SSE': [35733.97, 75556.39, 98089.77, 84080.12],
    'Tri-Ensemble SSE': [37322.54, 69324.10, 91070.22, 83031.40],
    'SSE Reduction (%)': ['+4.4%', '-8.2%', '-7.2%', '-1.2%'],
    'Baseline RMSLE': [1.064, 2.484, 2.604, 1.419],
    'Tri-Ensemble RMSLE': [1.088, 2.379, 2.509, 1.410],
})
sse_comp


In [ ]:
# Visualizing SSE Reduction across Transition States
fig, ax = plt.subplots(figsize=(10, 5))
states = sse_comp['Transition State'].to_list()
base_sse = sse_comp['Baseline SSE'].to_list()
tri_sse = sse_comp['Tri-Ensemble SSE'].to_list()

x = np.arange(len(states))
width = 0.35

ax.bar(x - width/2, base_sse, width, label='Baseline Clean Ensemble v4', color='#3498db', alpha=0.85)
ax.bar(x + width/2, tri_sse, width, label='Tri-Ensemble (CB + Hier-GRU + Transformer)', color='#2ecc71', alpha=0.85)

ax.set_ylabel('Sum of Squared Errors (SSE)')
ax.set_title('Снижение квадратичной ошибки (SSE) по 4-м переходам пользователей', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(states, fontsize=10)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
